# pré-traitement (réduction) des données capturées
 $\rightarrow$ **générer les DOF (dark, offset flat)**
 
 $\rightarrow$ **corriger les images de sciences des défauts du chemin optique, de la caméra et de l'atmosphère terrestre**



## définition des répertoires et fichiers

In [1]:
# répertoire de travail
#CAPTURE_DIR='data/20250807_deneb_ruchbah_gamcas/'
#CAPTURE_DIR='../../../CAPTURES/20250822_v1296Aql_altair_Tcrb_rsOph/'
#CAPTURE_DIR='../../../CAPTURES/20251102_cxdra_v1770cyg_deneb/'
CAPTURE_DIR = '../../../CAPTURES/20151031_agdra_n6543_sao17754/'


# bruts sciences
#CAPTURE_SCIENCE = "V1295Aql-300s-*.fits"
#CAPTURE_SCIENCE = "gamcas-5s-*.fit"
#CAPTURE_SCIENCE = "ruchbah-5s-*.fit"
#CAPTURE_SCIENCE = "lamCyg-60s-*.fits"
#CAPTURE_SCIENCE = "V1770Cyg-60s-*.fits"
CAPTURE_SCIENCE = "n6543-00*.fit"#"SAO17754-00*.fit"
#CAPTURE_SCIENCE = "m97-600s-*.fits"
#CAPTURE_SCIENCE = "sun-*.fits"

# bruts offset
CAPTURE_BIAS = "bias-00*.fit"

# bruts dark
#CAPTURE_DARK = "Dark-5s-*.fit"
#CAPTURE_DARK = "dark10m_DD200-0*.fit"
CAPTURE_DARK = "dark-00*.fit"

# bruts flat
#CAPTURE_FLAT = "flat_tung_-1s*.fit"
#CAPTURE_FLAT = "flat-0-5s-*.fits"
CAPTURE_FLAT = "flat-00*.fit"

# bruts néon
#CAPTURE_CALIB = "neon-30s-*.fit"s
#CAPTURE_CALIB = "neon2-5s-*.fits"
CAPTURE_CALIB = "neon_table-5s-210-*.fits"

# crop (vertical Y px) sinon None pour pas de crop 
TRIM_TOP = None #700 #None 
TRIM_BOTTOM = None #11250 #None

# fichiers de sortie
OUTPUT_BIAS = 'master_bias.fits'
OUTPUT_DARK = 'master_dark.fits'
OUTPUT_FLAT = 'master_flat.fits'
OUTPUT_CALIB = 'master_calib.fits'
OUTPUT_SCIENCE = 'master_science.fits'

import warnings, pathlib, os

# on affiche les fichiers 
for _type in (CAPTURE_BIAS, CAPTURE_DARK, CAPTURE_FLAT, CAPTURE_CALIB, CAPTURE_SCIENCE):
    print("\n",[f.name for f in pathlib.Path(CAPTURE_DIR).glob(_type)])



 ['bias-008.fit', 'bias-009.fit', 'bias-007.fit', 'bias-006.fit', 'bias-004.fit', 'bias-005.fit', 'bias-001.fit', 'bias-002.fit', 'bias-003.fit']

 ['dark-003.fit', 'dark-002.fit', 'dark-001.fit', 'dark-005.fit', 'dark-004.fit', 'dark-006.fit', 'dark-007.fit', 'dark-009.fit', 'dark-008.fit']

 ['flat-001.fit', 'flat-003.fit', 'flat-002.fit', 'flat-006.fit', 'flat-007.fit', 'flat-005.fit', 'flat-004.fit', 'flat-009.fit', 'flat-008.fit']

 []

 ['n6543-007.fit', 'n6543-006.fit', 'n6543-004.fit', 'n6543-005.fit', 'n6543-001.fit', 'n6543-002.fit', 'n6543-003.fit', 'n6543-008.fit', 'n6543-009.fit']


## création du dashboard
- lancer la cellule suivante
- souris sur '**Colormap**', bouton droit, menu "**create new view for cell output**"
- déplacer le nouvel onglet créé '**Output View**' pour le garder visible pendant que vous naviguez et exécutez les cellules de code

En cas de souci d'affichage $\rightarrow$  '**CTRL-R**'


In [2]:
%matplotlib widget
from image_widget import ImageWidget

db = ImageWidget()
db.show()


## import des libs 

In [3]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from astropy.nddata import CCDData
import ccdproc as ccdproc
import warnings, pathlib, os
from astropy.utils.exceptions import AstropyWarning, AstropyUserWarning
warnings.simplefilter('ignore', category=AstropyWarning)
warnings.simplefilter('ignore', category=AstropyUserWarning)
warnings.simplefilter('ignore', UserWarning)


# création du master bias

In [4]:
# charge toutes les images dans une liste de CCDData
CAPTURE_FILES = CAPTURE_BIAS
OUTPUT_FILE = OUTPUT_BIAS

img_list = [ccdproc.trim_image(CCDData.read(f, unit=u.Unit('adu'))[TRIM_TOP:TRIM_BOTTOM, :]) 
            for f in list(pathlib.Path(CAPTURE_DIR).glob(CAPTURE_FILES))]

# combine les images (sum, median, average, sigma_clip, trim, etc...)
img_combined = ccdproc.Combiner(img_list).median_combine()

# sauve l'image combinée
img_combined.write(CAPTURE_DIR + OUTPUT_FILE, overwrite=True) 

# affiche l'image
db.show_image(img_combined, name='master bias')


INFO: affichage de l'image master bias : bin=1, shape=(609, 809), min=1211, avg=1275.0, max=1361, stddev=17.7


# création du master dark

In [5]:
# nomme les images à combiner
CAPTURE_FILES = CAPTURE_DARK
OUTPUT_FILE = OUTPUT_DARK

# recharge l'offset combiné
master_bias = CCDData.read(CAPTURE_DIR + OUTPUT_BIAS, unit=u.Unit('adu'))

# charge toutes les images dans une liste de CCDData
img_list = [ccdproc.trim_image(CCDData.read(f, unit=u.Unit('adu'))[TRIM_TOP:TRIM_BOTTOM, :])
                               for f in list(pathlib.Path(CAPTURE_DIR).glob(CAPTURE_FILES))]

# récupére les temps d'intégration des dark qui sera utilisé plus tard
DARK_EXPTIME = img_list[0].meta['EXPTIME']

# supprime l'offset de chaque dark (car on va les 'scaler' plus tard -> il ne faut retirer l'offset)
img_corrected = [ccdproc.subtract_bias(ccd, master_bias) for ccd in img_list]

# combine les images (sum, median, average, sigma_clip, trim, etc...)
img_combined = ccdproc.Combiner(img_corrected).median_combine()

# mets à jour ses metadonnées à partir d'une image source
img_combined.meta = img_list[0].meta.copy()

# sauve l'image combinée
img_combined.write(CAPTURE_DIR + OUTPUT_FILE, overwrite=True) 

# affiche l'image
db.show_image(img_combined, name='master dark')



INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: affichage de l'image master dark : bin=1, shape=(609, 809), min=-90, avg=9.3, max=12230, stddev=66.3


# création du master flat

In [6]:
# nomme les images à combiner
CAPTURE_FILES = CAPTURE_FLAT
OUTPUT_FILE = OUTPUT_FLAT

# recharge l'offset combiné
master_bias = CCDData.read(CAPTURE_DIR + OUTPUT_BIAS, unit=u.Unit('adu'))

# charge toutes les images dans une liste de CCDData
img_list = [ccdproc.trim_image(CCDData.read(f, unit=u.Unit('adu'))[TRIM_TOP:TRIM_BOTTOM, :])
                               for f in list(pathlib.Path(CAPTURE_DIR).glob(CAPTURE_FILES))]

# supprime l'offset de chaque dark (car on va les 'scaler' plus tard -> il ne faut retirer l'offset)
img_corrected = [ccdproc.subtract_bias(ccd, master_bias) for ccd in img_list]

# combine les images (sum, median, average, sigma_clip, trim, etc...)
img_combined = ccdproc.Combiner(img_corrected).sum_combine() #median_combine()

# on remonte à > 0.0 et on normalise
min_val = np.min(img_combined)
epsilon = 1e-6  # pour éviter le zéro
img_combined = CCDData(img_combined.data - min_val + epsilon, unit=u.Unit('adu'))
img_combined = img_combined.divide(np.median(img_combined.data))

# mets à jour ses metadonnées à partir d'une image source
img_combined.meta = img_list[0].meta.copy()

# sauve l'image combinée
img_combined.write(CAPTURE_DIR + OUTPUT_FILE, overwrite=True) 

# affiche l'image
db.show_image(img_combined, name='master flat')



INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: affichage de l'image master flat : bin=1, shape=(609, 809), min=0, avg=1.0, max=2, stddev=0.2


# création du master calib (néon)

In [7]:
# nomme les images à combiner
CAPTURE_FILES = CAPTURE_CALIB
OUTPUT_FILE = OUTPUT_CALIB

# recharge l'offset combiné
master_bias = CCDData.read(CAPTURE_DIR + OUTPUT_BIAS, unit=u.Unit('adu'))

# charge toutes les images dans une liste de CCDData
img_list = [ccdproc.trim_image(CCDData.read(f, unit=u.Unit('adu'))[TRIM_TOP:TRIM_BOTTOM, :])
            for f in list(pathlib.Path(CAPTURE_DIR).glob(CAPTURE_FILES))]

# supprime l'offset de chaque dark (car on va les 'scaler' plus tard -> il ne faut retirer l'offset)
img_corrected = [ccdproc.subtract_bias(ccd, master_bias) for ccd in img_list]

# combine les images (sum, median, average, sigma_clip, trim, etc...)
img_combined = ccdproc.Combiner(img_corrected).median_combine()

# mets à jour ses metadonnées à partir d'une image source
img_combined.meta = img_list[0].meta.copy()

# sauve l'image combinée
img_combined.write(CAPTURE_DIR + OUTPUT_FILE, overwrite=True) 

# affiche l'image
db.show_image(img_combined, name='master calib')


INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]


TypeError: can only concatenate tuple (not "NoneType") to tuple

# création du master science

In [18]:
### fonction de recalage des spectres 2D
### astroalign ne fonctionne pas car pas d'étoiles
### --> on passe par une convolution FFT 

"""
align a set of loaded frames - specific to spectra fields (fft based)
"""
from scipy.signal import fftconvolve
def spec_align_v0(img_list, ref_image_index: int = 0):    
    ### Collect arrays and crosscorrelate all (except the first) with the first.
    print('align: fftconvolve running...')
    nX, nY = img_list[ref_image_index].shape
    correlations = [fftconvolve(img_list[ref_image_index].data.astype('float32'),
                                image[::-1, ::-1].data.astype('float32'),
                                mode='same') 
                    for image in img_list[1:]]

    ### For each image determine the coordinate of maximum cross-correlation.
    print('align: get max cross-correlation for every image...')
    shift_indices = [np.unravel_index(np.argmax(corr_array, axis=None), corr_array.shape) 
                     for corr_array in correlations]
    
    deltas = [(ind[0] - int(nX / 2), ind[1] - int(nY / 2)) for ind in shift_indices]
    for dt in deltas: print(f"align: images deltas = {dt[0]:.0f} / {dt[1]:.0f}")  

    ### Warn for ghost images if realignment requires shifting by more than 15% of the field size.
    x_frac = abs(max(deltas, key=lambda x: abs(x[0]))[0]) / nX
    y_frac = abs(max(deltas, key=lambda x: abs(x[1]))[1]) / nY
    t_frac = max(x_frac, y_frac)
    if t_frac > 0.15:
        print('align warning : shifting by {}% of the field size'.format(int(100 * t_frac)))

    ### Roll the images to realign them and return their median.
    print('align: images realignement ...')
    realigned_images = [CCDData(np.roll(image, deltas[i], axis=(0, 1)), unit = u.adu, header = image.header) 
                        for (i, image) in enumerate(img_list[1:])]

    ### do not forget the reference image
    realigned_images.append(CCDData(img_list[ref_image_index].data.astype('float32'), unit = u.adu, header = img_list[ref_image_index].header))
    print('align: complete')
    
    return (realigned_images)

from skimage.registration import phase_cross_correlation
import scipy.ndimage as ndimage
def spec_align(img_list, ref_image_index: int = 0):    
    """
    Align a set of loaded frames against a reference image.

    Uses phase_cross_correlation (scikit-image)

    Args:
        ref_image_index (int): index of the reference image (default: 0)

    Returns:
        ImagesCombiner: new combiner with all images shifted to the reference
    """
    if len(img_list) == 1:
        print('align: single image, nothing to align')
        return img_list

    ref_data = img_list[ref_image_index].data

    print(f'align: reference image index={ref_image_index}, shape={ref_data.shape}')

    realigned_images = []
    for i, image in enumerate(img_list):
        if i == ref_image_index:
            realigned_images.append(image)
            continue

        moving = image.data

        # phase_cross_correlation returns (shift, error, phasediff)
        shift, error, phasediff = phase_cross_correlation(
                    ref_data,
                    moving,
                    normalization=None
                    )

        dy, dx = float(shift[0]), float(shift[1])
        print(f'align: image #{i} shift = (dy={dy:.2f}, dx={dx:.2f} px), error={error:.2f}')

        # Warn if shift is large (> 5 % of field size)
        nY, nX = ref_data.shape
        if max(abs(dy) / nY, abs(dx) / nX) > 0.05:
            print(
                f'align: image #{i} requires a large shift '
                f'({abs(dy)/nY*100:.1f}% Y, {abs(dx)/nX*100:.1f}% X) — check acquisition'
            )

        # ndimage.shift fills borders with 0
        shifted = ndimage.shift(moving, shift=(dy, dx), order=1, cval=0.0)
        realigned_images.append(
            CCDData(shifted, unit=u.Unit('adu'), header=image.header)
        )

    print(f'align: {len(realigned_images)} images registered')
    return realigned_images



In [19]:
# nomme les images à combiner
CAPTURE_FILES = CAPTURE_SCIENCE
OUTPUT_FILE = OUTPUT_SCIENCE

# recharge les masters
master_bias = CCDData.read(CAPTURE_DIR + OUTPUT_BIAS, unit=u.Unit('adu'))
master_dark = CCDData.read(CAPTURE_DIR + OUTPUT_DARK, unit=u.Unit('adu'))
master_flat = CCDData.read(CAPTURE_DIR + OUTPUT_FLAT, unit=u.Unit('adu'))

# charge toutes les images dans une liste de CCDData
img_list = [ccdproc.trim_image(CCDData.read(f, unit=u.Unit('adu'))[TRIM_TOP:TRIM_BOTTOM, :])
            for f in list(pathlib.Path(CAPTURE_DIR).glob(CAPTURE_FILES))]

# on récupére le temps d'intégration des images sciences
SCIENCE_EXPTIME = img_list[0].meta['EXPTIME']
#print(f"{SCIENCE_EXPTIME=}, {DARK_EXPTIME=}")

# pour chaque image, on soustrait l'offset, scale et soustrait le dark et divise le flat 
sciences_corr = []
for img in img_list:    
    science_corr = ccdproc.subtract_bias(ccd=img, master=master_bias) 
    science_corr = ccdproc.subtract_dark(ccd=science_corr, master=master_dark, 
                                         scale=True, 
                                         data_exposure=SCIENCE_EXPTIME* u.Unit('s'), 
                                         dark_exposure=DARK_EXPTIME* u.Unit('s'),
                                         exposure_unit=u.Unit('s'))
    science_corr = ccdproc.flat_correct(ccd=science_corr, flat=master_flat)
  
    sciences_corr.append(science_corr)


# optionel : on recale les images prétraitées par rapport à une référence (la 1ere)
img_registered = spec_align(sciences_corr)
#img_registered = sciences_corr

# on combine les images (sum, median, average, sigma_clip, trim, etc...)
img_combined = ccdproc.Combiner(img_registered).sum_combine()    #sum_combine() #median_combine() 

# on met à jour les metadonnées à partir d'une image source
img_combined.meta = img_list[0].meta.copy()

# et enfin on sauve l'image combinée
img_combined.write(CAPTURE_DIR + OUTPUT_FILE, overwrite=True) 

# on affiche l'image
db.show_image(img_combined, name='master science')


INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
align: reference image index=0, shape=(609, 809)
align: image #1 shift = (dy=0.00, dx=0.00 px), error=0.00
align: image #2 shift = (dy=0.00, dx=0.00 px), error=0.00
align: image #3 shift = (dy=0.00, dx=0.00 px), error=0.00
align: image #4 shift = (dy=0.00, dx=0.00 px), error=0.00
align: image #5 shift = (dy=0.00, dx=0.00 px), error=0.00
align: image #6 shift = (dy=0.00, dx=0.00 px), error=0.00
align: image #7 shift = (dy=0.00, dx=0.00 px), error=0.00
align: image #8 shift = (dy=0.00, dx=0.00 px), error=0.00
align: 9 images registered
INFO: affichage de l'image master science : bin=1, shape=(609, 809), min=7333, avg=187854451.6, max=925457

# correction des cosmics


In [ ]:
# normalement pas nécessaire si combiner.median utilisé pour le master-science
# récupérer code de QuickSpec